# Explanation Notebook for ScenarioModel7
*Model Type:* **LogisticRegression**
*User:* UserAuto (Expertise: Beginner, Formats: [plainText], Details: low, Purpose: general)
*Explanation Method:* FeatureImportance *(auto-selected)*


## Training the LogisticRegression Model
We train a **LogisticRegression** model on the provided dataset.


In [1]:
import warnings; warnings.filterwarnings('ignore')
!pip install -q scikit-learn pandas matplotlib
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv(r"data/breast_cancer.csv")
y = df['target']
X = df.drop(columns=['target']).values
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    print('[Info] Used stratified split to preserve class proportions in train/test.')
except Exception:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    print('[Info] Used non-stratified split (stratification not applicable).')
class Dummy: pass
data = Dummy(); data.feature_names = list(df.drop(columns=['target']).columns)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
base_model = LogisticRegression(max_iter=1000, random_state=0)
model = make_pipeline(StandardScaler(), base_model)
print('[Info] Normalization enabled by training policy: using StandardScaler in a pipeline.')
model.fit(X_train, y_train)
print('Accuracy: LogisticRegression = ' + format(model.score(X_test, y_test), '.2f'))


[Info] Used stratified split to preserve class proportions in train/test.
[Info] Normalization enabled by training policy: using StandardScaler in a pipeline.
Accuracy: LogisticRegression = 0.98


## Explaining the model using FeatureImportance
Feature Importance ranks features by their overall influence on the model’s predictions.


In [2]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt

def _get_feature_names():
    try:
        names = list(getattr(data, 'feature_names', []))
        if names and len(names) == X_train.shape[1]:
            return names
    except Exception:
        pass
    if 'df' in globals():
        cols = [c for c in df.columns if c.lower() not in ('label','target','y')]
        if len(cols) == X_train.shape[1]:
            return cols
    return [f'f{j}' for j in range(X_train.shape[1])]

RAW_NAMES = _get_feature_names()
def _humanize(name):
    s = name.replace('_',' ').strip()
    s = re.sub(r'(?<!^)(?=[A-Z])',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return ' '.join([w.upper() if w.lower() in {'svm','pdp','ice','api','url'} else w.capitalize() for w in s.split(' ')])
HUMAN = [_humanize(n) for n in RAW_NAMES]

max_display = 3
table_rows = 6
curve_resolution = 12
ice_instances = 5
metric_digits = 2
figure_height = 3.8
line_width = 1.5
pdp_sample_rows = 6
ice_detailed_instances = 2
sampled_curve_points = 5
print('\n[Feature Importance overview]')
print('Feature importance is a global explanation: it ranks which features the model relies on most overall.')
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
elif hasattr(model, 'coef_'):
    coef = model.coef_
    importances = np.abs(coef[0]) if getattr(coef, 'ndim', 1) > 1 else np.abs(coef)
else:
    from sklearn.inspection import permutation_importance
    result = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=0)
    importances = result.importances_mean
pairs = sorted(list(zip(HUMAN, importances)), key=lambda x: abs(float(x[1])), reverse=True)[:max_display]
df_fi = pd.DataFrame({'Feature':[p[0] for p in pairs], 'Importance':[abs(float(p[1])) for p in pairs]})
print('\n[Plain-text explanation]')
print("This explanation summarizes the model's behaviour at a global level.")
print('The following features matter most to the model overall.')
for _, r in df_fi.iterrows():
    print(' - ' + str(r['Feature']) + ': importance=' + format(r['Importance'], '.4f'))



[Feature Importance overview]
Feature importance is a global explanation: it ranks which features the model relies on most overall.

[Plain-text explanation]
This explanation summarizes the model's behaviour at a global level.
The following features matter most to the model overall.
 - Mean Concave Points: importance=0.0298
 - Mean Concavity: importance=0.0281
 - Radius Error: importance=0.0281
